In [73]:
import torch
import torch.nn as nn
import numpy as np

In [74]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [75]:
import os
import json
import torch
import matplotlib.pyplot as plt
from PIL import Image
from collections import Counter

ANN_PATH = r"../data/iu_xray/annotation.json"
IMAGE_DIR = r"../data/iu_xray/images"

print(os.path.exists(ANN_PATH))
print(os.path.exists(IMAGE_DIR))
import sys
import os

# Lấy path project (folder cha của notebooks)
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Thêm vào Python path
sys.path.append(PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)

True
True
Project root: d:\HocTap\NCKH_ThayDoNhuTai\SGU_NCKH_dntai


In [76]:
with open(ANN_PATH, "r", encoding="utf-8") as f:
    ann = json.load(f)

print(ann.keys())

for split in ["train", "val", "test"]:
    print(split, len(ann[split]))

dict_keys(['train', 'val', 'test'])
train 2069
val 296
test 590


In [77]:
from modules import visual_extractor
class Args:
    image_dir = IMAGE_DIR
    ann_path = ANN_PATH
    dataset_name = "iu_xray"
    threshold = 3
    max_seq_length = 60
    visual_extractor = "resnet101"
    visual_extractor_pretrained = True
    batch_size = 64
    num_workers = 0
    d_model = 512
    d_ff = 512
    num_heads = 8
    num_layers = 3
    dropout = 0.1
    bos_idx = 0
    eos_idx = 0
    pad_idx = 0
    use_bn = 0
    drop_prob_lm = 0.5
    d_vf=2048
    cmm_size=2047
    cmm_dim=512
    topk=32
args = Args()


In [78]:
from modules.dataloaders import R2DataLoader
from modules.tokenizers import Tokenizer

tokenizer = Tokenizer(args)
print("vocab size:", len(tokenizer.token2idx))
print("first tokens:", list(tokenizer.token2idx.items())[:20])
train_loader = R2DataLoader(args, tokenizer, split="train", shuffle=False)

batch = next(iter(train_loader))

for i, x in enumerate(batch):
    if torch.is_tensor(x):
        print(i, x.shape, x.dtype)
    else:
        print(i, type(x), x)

vocab size: 760
first tokens: [('.', 1), ('1', 2), ('10', 3), ('10th', 4), ('12', 5), ('13', 6), ('17', 7), ('2', 8), ('3', 9), ('5', 10), ('5th', 11), ('6th', 12), ('7', 13), ('7th', 14), ('8', 15), ('8th', 16), ('9th', 17), ('<unk>', 18), ('a', 19), ('abdomen', 20)]
0 <class 'tuple'> ('CXR2384_IM-0942', 'CXR2926_IM-1328', 'CXR1451_IM-0291', 'CXR2887_IM-1289', 'CXR1647_IM-0424', 'CXR2434_IM-0976', 'CXR3655_IM-1817', 'CXR3299_IM-1575', 'CXR1419_IM-0267', 'CXR2867_IM-1274', 'CXR177_IM-0503', 'CXR3597_IM-1775', 'CXR2365_IM-0927', 'CXR718_IM-2280', 'CXR229_IM-0873', 'CXR1237_IM-0159', 'CXR2704_IM-1171', 'CXR1324_IM-0209', 'CXR3516_IM-1715', 'CXR3102_IM-1454', 'CXR1267_IM-0179', 'CXR3766_IM-1885', 'CXR1145_IM-0097', 'CXR1573_IM-0374', 'CXR2372_IM-0933-0001', 'CXR3115_IM-1463', 'CXR3706_IM-1851', 'CXR1616_IM-0399', 'CXR2845_IM-1254', 'CXR951_IM-2447', 'CXR3404_IM-1647', 'CXR3956_IM-2021', 'CXR1981_IM-0638', 'CXR930_IM-2429', 'CXR2269_IM-0858', 'CXR2328_IM-0898', 'CXR1167_IM-0112', 'CXR152_I

In [79]:
batch = next(iter(train_loader))

print(type(batch))
print("len batch:", len(batch))

ids, images, input_ids, masks = batch

print("ids type:", type(ids))
print("images shape:", images.shape)
print("input_ids shape:", input_ids.shape)
print("masks shape:", masks.shape)

print("images dtype:", images.dtype)
print("input_ids dtype:", input_ids.dtype)
print("masks dtype:", masks.dtype)

<class 'tuple'>
len batch: 4
ids type: <class 'tuple'>
images shape: torch.Size([64, 2, 3, 224, 224])
input_ids shape: torch.Size([64, 60])
masks shape: torch.Size([64, 60])
images dtype: torch.float32
input_ids dtype: torch.int64
masks dtype: torch.float32


In [80]:
images = images.to(device)
input_ids = input_ids.to(device)
masks = masks.to(device)

print("images device:", images.device)
print("input_ids device:", input_ids.device)
print("masks device:", masks.device)

images device: cuda:0
input_ids device: cuda:0
masks device: cuda:0


In [81]:
print("images min:", images.min().item())
print("images max:", images.max().item())
print("images mean:", images.mean().item())
print("images std:", images.std().item())

print("has NaN:", torch.isnan(images).any().item())
print("has Inf:", torch.isinf(images).any().item())

images min: -2.1179039478302
images max: 2.640000104904175
images mean: 0.3272649943828583
images std: 1.0959017276763916
has NaN: False
has Inf: False


In [82]:
print("input_ids min:", input_ids.min().item())
print("input_ids max:", input_ids.max().item())
print("unique first sample:", torch.unique(input_ids[0]))

print("mask min:", masks.min().item())
print("mask max:", masks.max().item())
print("mask sum first sample:", masks[0].sum().item())

input_ids min: 0
input_ids max: 758
unique first sample: tensor([  0,   1,  19,  40,  49,  62,  68, 143, 198, 225, 239, 284, 293, 314,
        318, 319, 341, 364, 375, 388, 406, 445, 451, 455, 467, 475, 517, 521,
        534, 550, 604, 624, 639, 684, 725, 752], device='cuda:0')
mask min: 0.0
mask max: 1.0
mask sum first sample: 45.0


In [83]:
from models.models import BaseCMNModel

model = BaseCMNModel(args, tokenizer)

print(model)

BaseCMNModel(
  (visual_extractor): VisualExtractor(
    (model): Sequential(
      (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (4): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu): ReLU(inpl

In [84]:
model = model.to(device)
print("Model loaded to:", device)

Model loaded to: cuda


In [ ]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Total params:", total_params)
print("Trainable params:", trainable_params)
print("Frozen params:", total_params - trainable_params)

Total params: 59052857
Trainable params: 59052857
Frozen params: 0


: 